In [1]:
# Resolve paths consistently when launched from the root or notebooks/.
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "topic_embeddings").is_dir() and (p / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch this notebook from within the derStandard project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)


# Embedding grid search: explicit PCA and SGNS

Train every combination in the grids below using the same reusable functions imported by `2_explicit_embeddings.ipynb` and `2_sgns_embeddings.ipynb`. Evaluate with the shared implementations from `3_topic_analogies.ipynb` and `3_topic_related.ipynb`.

The final summary contains all effective hyperparameters and five **accuracy percentages (0–100; higher is better)**:

- Analogies: top-1, top-3 and top-5 hit rates over all analogy rows.
- Close themes: average of a_to_b and b_to_a top-5 hit rates.
- Taxonomy relocations: average of a_to_b and b_to_a top-5 hit rates.

Missing topics, path mismatches and other per-example validation errors count as misses, as in the evaluation notebooks. Failed model runs have blank metrics and an error message. The evaluation set stays fixed across models. These are tuning results; reserve independent examples to estimate generalization.


## Imports


In [ ]:
import hashlib
import itertools
import json
from pathlib import Path
import platform
import time
import traceback

import numpy as np
import pandas as pd
import scipy
from scipy.sparse import load_npz, save_npz
import sklearn

from topic_embeddings import explicit, sgns
from topic_embeddings.evaluate_analogies import AnalogyEvaluator
from topic_embeddings.evaluate_related_topics import RelatedEvaluator, summarize_related_main

TOPIC_COLUMNS = ["Topic_1", "Topic_2", "Topic_3"]


## Load data functions

In [3]:
def load_topics_articles(data_dir):
    data_dir = Path(data_dir)
    topics = pd.read_csv(data_dir / "top_1000_topics.csv", dtype=str, keep_default_na=False)
    articles = pd.read_csv(data_dir / "top_1000_articles.csv", dtype=str, keep_default_na=False,
                           usecols=["Article_Id", "Topic_Id", *TOPIC_COLUMNS])
    for frame, key in ((topics, "Topic_Id"), (articles, "Article_Id")):
        if frame[key].eq("").any() or frame[key].duplicated().any():
            raise ValueError(f"{key} must be nonempty and unique")
    if not articles.Topic_Id.isin(topics.Topic_Id).all():
        raise ValueError("Article references an unknown topic ID")
    return topics, articles


def comment_paths(data_dir, raw_data_dir="data/raw", use_raw_comments=False):
    if use_raw_comments:
        paths = sorted(Path(raw_data_dir).glob("comments_file_*_[0-9][0-9].tsv"))
        if not paths:
            raise FileNotFoundError("No raw comment shards found")
        return paths
    return [Path(data_dir) / "top_1000_comments.csv"]


def read_comment_chunks(paths, chunk_size=500_000, use_raw_comments=False):
    for path in paths:
        with pd.read_csv(path, sep="\t" if use_raw_comments else ",", dtype=str,
                         keep_default_na=False, usecols=["Article_Id", "Author_Id"],
                         chunksize=chunk_size) as reader:
            yield from reader


def load_evaluation(evaluation_dir):
    analogies = pd.read_csv(evaluation_dir / "topic_analogy_candidates.csv",
                            dtype=str, keep_default_na=False)
    pairs = pd.read_csv(evaluation_dir / "topic_related_candidates.csv",
                       dtype=str, keep_default_na=False)
    required_relations = {"close_theme", "taxonomy_relocation"}
    if analogies.empty or not required_relations.issubset(set(pairs.relation)):
        raise ValueError("Evaluation needs analogies and both related-topic relation groups")
    return analogies, pairs

## Evaluation metrics


In [4]:
METRIC_COLUMNS = ["analogy_top1_accuracy", "analogy_top3_accuracy", "analogy_top5_accuracy",
                  "close_theme_top5_accuracy", "taxonomy_relocation_top5_accuracy"]


def evaluate_embeddings(embeddings, analogies, pairs):
    """Return five percentages plus per-example results. Denominators include errors."""
    analogy_results = AnalogyEvaluator(embeddings, analogies).evaluate_all_analogies(analogies)
    pair_results = RelatedEvaluator(embeddings, pairs).evaluate_all_pairs(pairs)
    related = summarize_related_main(pair_results).set_index("Relation")["Average top-5 hit rate (%)"]
    metrics = {f"analogy_top{k}_accuracy": float(100 * analogy_results[f"top{k}"].mean())
               for k in (1, 3, 5)}
    metrics.update(close_theme_top5_accuracy=float(related["Close themes"]),
                   taxonomy_relocation_top5_accuracy=float(related["Taxonomy relocations"]))
    return metrics, analogy_results, pair_results


def evaluate_and_save(frame, analogies, pairs, run_dir):
    """Persist detailed evaluations and return the fields for the run summary."""
    metrics, analogy_results, pair_results = evaluate_embeddings(frame, analogies, pairs)
    analogy_results.to_csv(run_dir / "analogy_results.csv", index=False)
    pair_results.to_csv(run_dir / "related_results.csv", index=False)
    return dict(
        **metrics, retained_topics=len(frame),
        analogy_errors=int(analogy_results.error.ne("").sum()),
        related_errors=int(pair_results.error.ne("").sum()),
        embeddings=str((run_dir / "topic_embeddings_normalized.csv").resolve()),
    )


## Grid defaults and validation


Only for validation, a well defined grid should not have to be expanded.

In [5]:
DEFAULTS = {
    "explicit_pca": dict(cutoff=5, context_dimensions=500, dimensions=150, smoothing=1.0, whiten=False),
    "sgns": dict(dimensions=150, negative=35, alpha=0.18, sample=0.0043,
                 epochs=1, threads=4, shuffle_seed=42),
}


def expand_grid(grids):
    """Fill omitted parameters with notebook defaults and expand every combination."""
    jobs = []
    for method, axes in grids.items():
        if method not in DEFAULTS:
            raise ValueError(f"Unknown method: {method}")
        if not isinstance(axes, dict) or set(axes) - set(DEFAULTS[method]):
            raise ValueError(f"Unknown parameters for {method}")
        if any(not isinstance(v, list) or not v for v in axes.values()):
            raise ValueError("Each grid axis must be a nonempty list")
        for values in itertools.product(*axes.values()):
            params = DEFAULTS[method] | dict(zip(axes, values))
            for key, value in params.items():
                if key == "whiten":
                    valid = isinstance(value, bool)
                elif key in ("smoothing", "alpha", "sample"):
                    valid = (isinstance(value, (float, int)) and not isinstance(value, bool)
                             and np.isfinite(value) and (value >= 0 if key == "sample" else value > 0))
                else:
                    valid = (isinstance(value, int) and not isinstance(value, bool)
                             and (value >= 0 if key in ("cutoff", "shuffle_seed") else value > 0))
                if not valid:
                    raise ValueError(f"Invalid {method}.{key}: {value}")
            if (method, params) not in jobs:
                jobs.append((method, params))
    if not jobs:
        raise ValueError("No combinations to try")
    return jobs


## Cache and per-method training

Explicit models share one topic–user count matrix. SGNS models share prepared pairs for each shuffle seed. Each training helper exports embeddings and returns the normalized table for evaluation.

In [6]:
def load_or_count_users(cache, paths, article_to_index, n_topics, chunk_size, use_raw_comments):
    """Read comments once for the entire explicit grid; reuse counts on resume."""
    destination = cache / "counts.npz"
    if destination.exists():
        return load_npz(destination)
    counts, stats = explicit.count_topic_users(
        read_comment_chunks(paths, chunk_size, use_raw_comments),
        article_to_index, n_topics, verbose=True)
    temporary = cache / "counts.tmp.npz"
    save_npz(temporary, counts)
    temporary.replace(destination)
    atomic_json(cache / "counting_stats.json", stats)
    return counts


def train_explicit_run(params, counts, topics, run_dir):
    """Train and export one PCA model using the package's explicit API."""
    retained, raw, normalized, _ = explicit.train_explicit(counts, **params)
    metadata = topics.iloc[retained]
    explicit.export_embeddings(run_dir, metadata, raw, normalized)
    return explicit.embedding_frame(metadata, normalized)


def train_sgns_run(params, trainer, cache, paths, topics, article_to_topic,
                   run_dir, chunk_size, use_raw_comments):
    """Reuse shuffled pairs by seed, then train only the topic vectors needed here."""
    seed = params["shuffle_seed"]
    prepared = cache / f"sgns_pairs_{seed}"
    prepared.mkdir(exist_ok=True)
    required = ["ready.json", "pairs.txt", "topics.vocab", "users.vocab"]
    if not all((prepared / name).is_file() for name in required):
        stats, _ = sgns.prepare_training_pairs(
            read_comment_chunks(paths, chunk_size, use_raw_comments), topics,
            article_to_topic, prepared, seed, chunk_size)
        atomic_json(prepared / "ready.json", stats)
    training_params = {key: value for key, value in params.items() if key != "shuffle_seed"}
    sgns.train_prepared(trainer, prepared, run_dir, dump_context=False, **training_params)
    tokens = [f"t{i}" for i in range(len(topics))]
    raw = sgns.read_vectors(run_dir / "topic_vectors.txt", tokens)
    normalized, zero_rows = sgns.normalize_rows(raw)
    if len(zero_rows):
        raise ValueError("SGNS produced zero topic vectors")
    metadata = topics.copy()
    metadata[TOPIC_COLUMNS] = metadata[TOPIC_COLUMNS].apply(lambda column: column.str.strip())
    # SGNS export_embeddings also requires user vectors, which grid search skips.
    frame = sgns.embedding_frame(metadata, normalized)
    frame.to_csv(run_dir / "topic_embeddings_normalized.csv", index=False)
    return frame

## Results and resume checks

Run IDs depend on the method and effective parameters. The manifest prevents reuse after input, implementation, environment, or chunk-size changes.

In [7]:
def atomic_json(path, value):
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def best_models(results):
    """Best successful model for each requested metric; ties use run ID."""
    successful = results.loc[results.status == "ok"]
    rows = []
    for metric in METRIC_COLUMNS:
        if not successful.empty:
            best = successful.sort_values([metric, "run_id"], ascending=[False, True]).iloc[0]
            rows.append(dict(metric=metric, accuracy_pct=best[metric], run_id=best.run_id,
                             method=best.method, embeddings=best.embeddings))
    return pd.DataFrame(rows, columns=["metric", "accuracy_pct", "run_id", "method", "embeddings"])


def save_summary(output_dir, records):
    table = pd.DataFrame(records)
    temporary = output_dir / "results.tmp.csv"
    table.to_csv(temporary, index=False)
    temporary.replace(output_dir / "results.csv")
    winners = best_models(table)
    temporary = output_dir / "best_models.tmp.csv"
    winners.to_csv(temporary, index=False)
    temporary.replace(output_dir / "best_models.csv")
    return table

In [8]:
def build_manifest(sources, jobs, tool_dir, use_raw_comments, chunk_size):
    """Identify inputs and implementation before reusing any cached work."""
    manifest = dict(
        inputs=[dict(path=str(p.resolve()), size=p.stat().st_size, mtime_ns=p.stat().st_mtime_ns)
                for p in sources],
        code={p.name: hashlib.sha256(p.read_bytes()).hexdigest()
              for p in Path("topic_embeddings").glob("*.py")},
        versions=dict(python=platform.python_version(), numpy=np.__version__,
                      pandas=pd.__version__, scipy=scipy.__version__, sklearn=sklearn.__version__),
        methods=sorted({method for method, _ in jobs}),
        tool_commit=sgns.TOOL_COMMIT, use_raw_comments=use_raw_comments,
        chunk_size=chunk_size,
    )
    # Settings and outputs may change; every implementation cell participates.
    notebook = json.loads(Path("notebooks/4_embedding_grid_search.ipynb").read_text())
    implementation = ["".join(cell["source"]) for cell in notebook["cells"]
                      if "grid-search-implementation" in cell.get("metadata", {}).get("tags", [])]
    manifest["code"]["4_embedding_grid_search.ipynb"] = hashlib.sha256(
        json.dumps(implementation).encode()).hexdigest()
    if "sgns" in manifest["methods"]:
        manifest["trainer_sources"] = {
            name: hashlib.sha256((Path(tool_dir) / name).read_bytes()).hexdigest()
            for name in ("word2vecf.c", "vocab.c", "io.c", "vocab.h", "io.h")
        }
    return manifest


def check_manifest(output_dir, manifest):
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / "manifest.json"
    if path.exists() and json.loads(path.read_text()) != manifest:
        raise ValueError("Inputs, code or environment changed; choose a new OUTPUT_DIR")
    atomic_json(path, manifest)


def reusable_result(path, retry_failed):
    """Return a completed result, or a failed result unless retries are enabled."""
    if not path.exists():
        return None
    previous = json.loads(path.read_text())
    complete = previous["status"] == "ok" and Path(previous["embeddings"]).is_file()
    if complete or (previous["status"] == "failed" and not retry_failed):
        return previous
    return None

## Search runner

For each combination: reuse an existing result, or train, evaluate, and record success or failure. Completed runs survive an interruption.

In [9]:
def run_grid_search(grids, data_dir="data/processed", evaluation_dir="data/benchmarks",
                    output_dir="results/grid_search/current", tool_dir=".tools/word2vecf",
                    use_raw_comments=False, raw_data_dir="data/raw", chunk_size=500_000,
                    retry_failed=False):
    """Train, evaluate and persist every attempt; reuse compatible results on rerun."""
    # Expand the parameter lists into validated jobs, one per model configuration
    jobs = expand_grid(grids)
    data_dir, evaluation_dir, output_dir = map(Path, (data_dir, evaluation_dir, output_dir))
    paths = comment_paths(data_dir, raw_data_dir, use_raw_comments)

    # Refuse to reuse cached work if inputs, implementation, or environment changed
    sources = [data_dir / "top_1000_topics.csv", data_dir / "top_1000_articles.csv", *paths,
               evaluation_dir / "topic_analogy_candidates.csv", evaluation_dir / "topic_related_candidates.csv"]
    manifest = build_manifest(sources, jobs, tool_dir, use_raw_comments, chunk_size)
    check_manifest(output_dir, manifest)

    # Create cache dir
    cache = output_dir / "cache"
    cache.mkdir(exist_ok=True)

    # SGNS uses topic IDs; explicit models use row indices in the topic-user matrix
    topics, articles = load_topics_articles(data_dir)
    article_to_topic = articles.set_index("Article_Id")["Topic_Id"]
    article_to_index = article_to_topic.map(dict(zip(topics.Topic_Id, range(len(topics)))))

    # Every model is scored against the same fixed benchmarks.
    analogies, pairs = load_evaluation(evaluation_dir)
    counts, trainer = None, None
    records = []
    print(f"Grid search: {len(jobs)} combinations", flush=True)

    # Run models sequentially, checkpointing each completed attempt
    for number, (method, params) in enumerate(jobs, 1):
        # Stable IDs let the same effective parameters find their previous results
        key = hashlib.sha256(json.dumps(params, sort_keys=True).encode()).hexdigest()[:16]
        run_id = f"{method}-{key}"
        run_dir = output_dir / "runs" / run_id
        run_dir.mkdir(parents=True, exist_ok=True)
        result_path = run_dir / "result.json"

        # Reuse successful runs for re-runs
        previous = reusable_result(result_path, retry_failed)
        if previous is not None:
            records.append(previous)
            save_summary(output_dir, records)
            print(f"[{number}/{len(jobs)}] Reused {run_id}", flush=True)
            continue

        print(f"[{number}/{len(jobs)}] {run_id}: {params}", flush=True)
        start = time.monotonic()
        result = dict(run_id=run_id, method=method, **params, status="failed", error="",
                      **{metric: None for metric in METRIC_COLUMNS}, embeddings="")
        try:
            if method == "explicit_pca":
                # Count comments once, then reuse the sparse matrix for every PCA setting
                if counts is None:
                    counts = load_or_count_users(
                        cache, paths, article_to_index, len(topics), chunk_size, use_raw_comments)
                frame = train_explicit_run(params, counts, topics, run_dir)
            else:
                # Build the executable once; the helper also caches pairs by shuffle seed
                if trainer is None:
                    trainer = sgns.build_trainer(output_dir=cache, tool_dir=tool_dir)
                frame = train_sgns_run(
                    params, trainer, cache, paths, topics, article_to_topic,
                    run_dir, chunk_size, use_raw_comments)

            # Both training helpers return normalized topic embeddings in the same format
            result.update(evaluate_and_save(frame, analogies, pairs, run_dir))
            result["status"] = "ok"
        except Exception as exc:
            # Record this model's failure and continue with the remaining configurations
            result["error"] = f"{type(exc).__name__}: {exc}"
            (run_dir / "error.log").write_text(traceback.format_exc())
            print(f"Failed: {result['error']}", flush=True)

        # Persist successes and failures immediately so completed work survives interruption
        result["seconds"] = time.monotonic() - start
        atomic_json(result_path, result)
        records.append(result)
        save_summary(output_dir, records)

    # Summarize the current grid, including reused results and failed attempts.
    table = save_summary(output_dir, records)
    print(f"Finished: {(table.status == 'ok').sum()} succeeded, "
          f"{(table.status == 'failed').sum()} failed. Results: {output_dir / 'results.csv'}")
    return table

## Paths and runtime settings

In [ ]:
DATA_DIR = Path("data/processed")
EVALUATION_DIR = Path("data/benchmarks")
OUTPUT_DIR = Path("results/grid_search/current")
TOOL_DIR = Path(".tools/word2vecf")
RAW_DATA_DIR = Path("data/raw")
USE_RAW_COMMENTS = False
CHUNK_SIZE = 500_000
RETRY_FAILED = False


## Configure the grids

Each list is a search axis; all combinations are tried. The default is **16 explicit PCA + 32 SGNS = 48 models**. Reduce the lists for a shorter search, or omit one method entirely. Unspecified parameters use the training notebooks' defaults.

Explicit PCA always normalizes before and after centered full-SVD PCA. The grid varies the strict comment cutoff (`count > cutoff`), context count, PCA dimension, and additive PPMI smoothing. `whiten` can also be varied.

SGNS varies dimension, negative samples, learning rate, subsampling and epochs. A fixed shuffle seed shares the same globally shuffled training pairs across models. The grid uses one thread for repeatability; the standalone SGNS notebook retains four threads. The shuffle seed does not set the trainer's internal initialization. Multiple threads can yield slightly different results.


In [11]:
GRIDS = {
    "explicit_pca": {
        "cutoff": [0, 2, 5],
        "context_dimensions": [250, 500],
        "dimensions": [75, 150],
        "smoothing": [0.1, 1.0],
        "whiten": [False],
    },
    "sgns": {
        "dimensions": [75, 150],
        "negative": [5, 15, 35],
        "alpha": [0.025, 0.075, 0.18],
        "sample": [0, 0.001, 0.0043],
        "epochs": [1],
        "threads": [4], # makes it non deterministic, but faster
        "shuffle_seed": [42],
    },
}

planned_runs = pd.DataFrame([{"method": method, **params}
                             for method, params in expand_grid(GRIDS)])
print(f"Planned models: {len(planned_runs)}")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(planned_runs)


Planned models: 78


,method,cutoff,context_dimensions,dimensions,smoothing,whiten,negative,alpha,sample,epochs,threads,shuffle_seed
0,explicit_pca,0.0,250.0,75,0.1,False,NaN,NaN,NaN,NaN,NaN,NaN
1,explicit_pca,0.0,250.0,75,1.0,False,NaN,NaN,NaN,NaN,NaN,NaN
2,explicit_pca,0.0,250.0,150,0.1,False,NaN,NaN,NaN,NaN,NaN,NaN
3,explicit_pca,0.0,250.0,150,1.0,False,NaN,NaN,NaN,NaN,NaN,NaN
4,explicit_pca,0.0,500.0,75,0.1,False,NaN,NaN,NaN,NaN,NaN,NaN
5,explicit_pca,0.0,500.0,75,1.0,False,NaN,NaN,NaN,NaN,NaN,NaN
6,explicit_pca,0.0,500.0,150,0.1,False,NaN,NaN,NaN,NaN,NaN,NaN
7,explicit_pca,0.0,500.0,150,1.0,False,NaN,NaN,NaN,NaN,NaN,NaN
8,explicit_pca,2.0,250.0,75,0.1,False,NaN,NaN,NaN,NaN,NaN,NaN
9,explicit_pca,2.0,250.0,75,1.0,False,NaN,NaN,NaN,NaN,NaN,NaN


## Run or resume

This cell trains on all supplied comments, without sampling. The current dataset has about 49 million pairs, so the full search can take substantial time. Counts and shuffled SGNS pairs are cached on disk under `OUTPUT_DIR/cache`; each shuffle seed creates another pair file. SGNS requires the pinned `.tools/word2vecf` checkout and `cc`, as in the training notebook.

Each model gets its own directory containing normalized topic embeddings, detailed evaluations, parameters, status and timing. `results.csv` and `best_models.csv` update after every combination. Training failures are recorded and the search continues. Rerun to resume; set `RETRY_FAILED = True` to retry failures. Interrupting a cell preserves completed runs.

Use a new output directory after changing data, implementation cells, module code, package versions, chunk size, or the set of methods. Hyperparameter lists can be extended  and resume in the same directory. Input identity uses file paths, sizes and modification times; the manifest also records notebook implementation, module and trainer source hashes. Do not run two searches in the same output directory concurrently. Cached pairs can be large; remove the cache when you no longer need to train additional models.

Save notebook edits and rerun the implementation cells before resuming. This refactor changes the manifest, so use a fresh `OUTPUT_DIR` for results produced by the earlier notebook.


In [12]:
results = run_grid_search(
    GRIDS, data_dir=DATA_DIR, evaluation_dir=EVALUATION_DIR,
    output_dir=OUTPUT_DIR, tool_dir=TOOL_DIR,
    use_raw_comments=USE_RAW_COMMENTS, raw_data_dir=RAW_DATA_DIR,
    chunk_size=CHUNK_SIZE, retry_failed=RETRY_FAILED,
)


Grid search: 78 combinations
[1/78] explicit_pca-ea121337df63f144: {'cutoff': 0, 'context_dimensions': 250, 'dimensions': 75, 'smoothing': 0.1, 'whiten': False}
Processed 5,000,000 comments; 72,255 authors.
Processed 10,000,000 comments; 91,664 authors.
Processed 15,000,000 comments; 106,734 authors.
Processed 20,000,000 comments; 120,925 authors.
Processed 25,000,000 comments; 132,821 authors.
Processed 30,000,000 comments; 142,556 authors.
Processed 35,000,000 comments; 151,711 authors.
Processed 40,000,000 comments; 160,490 authors.
Processed 45,000,000 comments; 170,185 authors.
[2/78] explicit_pca-c91118438e21e3a0: {'cutoff': 0, 'context_dimensions': 250, 'dimensions': 75, 'smoothing': 1.0, 'whiten': False}
[3/78] explicit_pca-9b9f45df61effa71: {'cutoff': 0, 'context_dimensions': 250, 'dimensions': 150, 'smoothing': 0.1, 'whiten': False}
[4/78] explicit_pca-638a6634e293bf7f: {'cutoff': 0, 'context_dimensions': 250, 'dimensions': 150, 'smoothing': 1.0, 'whiten': False}
[5/78] expli

## Final summary: every combination and all five metrics

Change `SORT_BY` to any metric column to rank models for that task. Failed runs remain visible at the bottom. All values are percentages; for example, 40 means a 40% hit rate. There is no implicit combined score that gives extra weight to analogies just because they have three cutoffs.


In [13]:
SORT_BY = "analogy_top5_accuracy"
if SORT_BY not in METRIC_COLUMNS:
    raise ValueError(f"SORT_BY must be one of {METRIC_COLUMNS}")
summary = results.sort_values(SORT_BY, ascending=False, na_position="last", kind="stable")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(summary)
print(f"Every attempted combination: {(OUTPUT_DIR / 'results.csv').resolve()}")


,run_id,method,cutoff,context_dimensions,dimensions,smoothing,whiten,status,error,analogy_top1_accuracy,analogy_top3_accuracy,analogy_top5_accuracy,close_theme_top5_accuracy,taxonomy_relocation_top5_accuracy,embeddings,retained_topics,analogy_errors,related_errors,seconds,negative,alpha,sample,epochs,threads,shuffle_seed
73,sgns-574da586e454b52d,sgns,NaN,NaN,150,NaN,NaN,ok,,43.930636,65.895954,72.832370,44.749596,46.153846,/Users/nachodoradollamas/Downloads/derStandard...,1000,0,0,125.244786,35.0,0.075,0.0010,1.0,4.0,42.0
64,sgns-e84c7cb9f4a8a24b,sgns,NaN,NaN,150,NaN,NaN,ok,,45.086705,67.630058,71.676301,43.134087,40.384615,/Users/nachodoradollamas/Downloads/derStandard...,1000,0,0,75.268330,15.0,0.075,0.0010,1.0,4.0,42.0
76,sgns-cfda9d266fc3b6eb,sgns,NaN,NaN,150,NaN,NaN,ok,,43.352601,64.739884,71.676301,44.103393,50.000000,/Users/nachodoradollamas/Downloads/derStandard...,1000,0,0,127.159262,35.0,0.180,0.0010,1.0,4.0,42.0
77,sgns-a5e2ca952e63b819,sgns,NaN,NaN,150,NaN,NaN,ok,,46.242775,64.739884,71.098266,43.457189,46.153846,/Users/nachodoradollamas/Downloads/derStandard...,1000,0,0,161.500060,35.0,0.180,0.0043,1.0,4.0,42.0
75,sgns-b5f89509a0c30514,sgns,NaN,NaN,150,NaN,NaN,ok,,46.242775,65.317919,70.520231,42.972536,48.076923,/Users/nachodoradollamas/Downloads/derStandard...,1000,0,0,182.299593,35.0,0.180,0.0000,1.0,4.0,42.0
55,sgns-923bfaefc525930e,sgns,NaN,NaN,150,NaN,NaN,ok,,38.728324,62.427746,69.942197,42.649435,40.384615,/Users/nachodoradollamas/Downloads/derStandard...,1000,0,0,35.254464,5.0,0.075,0.0010,1.0,4.0,42.0
58,sgns-422643098e8600e0,sgns,NaN,NaN,150,NaN,NaN,ok,,42.774566,63.005780,69.942197,42.164782,26.923077,/Users/nachodoradollamas/Downloads/derStandard...,1000,0,0,37.714052,5.0,0.180,0.0010,1.0,4.0,42.0
63,sgns-1b1b37fb7f3b30e8,sgns,NaN,NaN,150,NaN,NaN,ok,,44.508671,64.161850,69.942197,44.022617,38.461538,/Users/nachodoradollamas/Downloads/derStandard...,1000,0,0,99.376146,15.0,0.075,0.0000,1.0,4.0,42.0
67,sgns-d2dbc9da2eb31807,sgns,NaN,NaN,150,NaN,NaN,ok,,43.352601,62.427746,69.364162,43.053312,44.230769,/Users/nachodoradollamas/Downloads/derStandard...,1000,0,0,67.747301,15.0,0.180,0.0010,1.0,4.0,42.0
74,sgns-9d84dc8a8170882d,sgns,NaN,NaN,150,NaN,NaN,ok,,43.930636,65.895954,69.364162,43.941842,46.153846,/Users/nachodoradollamas/Downloads/derStandard...,1000,0,0,161.772043,35.0,0.075,0.0043,1.0,4.0,42.0


Every attempted combination: /Users/nachodoradollamas/Downloads/derStandard/results/grid_search/current/results.csv


## Best embeddings for each metric

Each row links a metric to its best successful model and saved embeddings. Ties are resolved by run ID. Use the embedding path as `DATA_PATH` in the evaluation notebooks.


In [14]:
winners = best_models(results)
display(winners)


,metric,accuracy_pct,run_id,method,embeddings
0,analogy_top1_accuracy,46.820809,sgns-ad4af030cc3f2999,sgns,/Users/nachodoradollamas/Downloads/derStandard...
1,analogy_top3_accuracy,67.630058,sgns-e84c7cb9f4a8a24b,sgns,/Users/nachodoradollamas/Downloads/derStandard...
2,analogy_top5_accuracy,72.832370,sgns-574da586e454b52d,sgns,/Users/nachodoradollamas/Downloads/derStandard...
3,close_theme_top5_accuracy,44.749596,sgns-574da586e454b52d,sgns,/Users/nachodoradollamas/Downloads/derStandard...
4,taxonomy_relocation_top5_accuracy,50.000000,sgns-cfda9d266fc3b6eb,sgns,/Users/nachodoradollamas/Downloads/derStandard...
